# Confronto OCSE

Confronti OCSE su entrate e spesa. Il notebook legge la sezione `confronto_ocse` e mostra le viste disponibili nel payload.

In [ ]:
from pathlib import Path
import json
import pandas as pd
import matplotlib.pyplot as plt

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "data").exists() and PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

SOURCE_JSON = PROJECT_ROOT / "data/export/bilancio-pubblico/source-data.json"
SECTION_DIR = PROJECT_ROOT / "data/export/bilancio-pubblico/sections"

def load_section(section_id):
    section_path = SECTION_DIR / f"{section_id}.json"
    if section_path.exists():
        payload = json.loads(section_path.read_text(encoding="utf-8"))
        return payload.get("section", payload)
    payload = json.loads(SOURCE_JSON.read_text(encoding="utf-8"))
    return payload["sections"][section_id]

def frame(rows):
    return pd.DataFrame(rows or [])

def plot_bar(data, x, y, title, top=15):
    df = data.dropna(subset=[y]).sort_values(y, ascending=False).head(top).sort_values(y)
    ax = df.plot.barh(x=x, y=y, legend=False, figsize=(9, max(4, len(df) * 0.35)))
    ax.set_title(title)
    ax.set_xlabel(y)
    ax.set_ylabel("")
    plt.tight_layout()
    return ax

def plot_line(data, x, y, title):
    df = data.dropna(subset=[x, y]).sort_values(x)
    ax = df.plot.line(x=x, y=y, marker="o", legend=False, figsize=(9, 4.8))
    ax.set_title(title)
    ax.set_xlabel("")
    ax.set_ylabel(y)
    plt.tight_layout()
    return ax

ocse = load_section("confronto_ocse")
ocse.keys()

## Viste disponibili

Le viste OCSE sono separate dal confronto europeo perché fonte e perimetro sono diversi.

In [ ]:
display(pd.DataFrame(ocse["available_views"]))
data = ocse["data"]
data.keys()

## Pressione fiscale totale

Classifica dei paesi disponibili nel payload OCSE.

In [ ]:
tax = frame(data.get("peer_revenue_total"))
display(tax.head(20))
candidate_cols = [col for col in ["country", "paese", "code", "codice"] if col in tax.columns]
value_cols = [col for col in ["value", "valore", "percent_gdp"] if col in tax.columns]
if candidate_cols and value_cols:
    chart = tax.rename(columns={candidate_cols[0]: "country", value_cols[0]: "value"})
    plot_bar(chart, "country", "value", "Pressione fiscale totale OCSE")

## Spesa pubblica totale

Classifica dei paesi disponibili nel payload OCSE.

In [ ]:
spending = frame(data.get("peer_spending_total"))
display(spending.head(20))
candidate_cols = [col for col in ["country", "paese", "code", "codice"] if col in spending.columns]
value_cols = [col for col in ["value", "valore", "percent_gdp"] if col in spending.columns]
if candidate_cols and value_cols:
    chart = spending.rename(columns={candidate_cols[0]: "country", value_cols[0]: "value"})
    plot_bar(chart, "country", "value", "Spesa pubblica totale OCSE")

## Entrate e spese per categoria

Queste tabelle servono per capire quali categorie sono disponibili prima di costruire grafici più specifici.

In [ ]:
display(frame(data.get("revenue_categories")).head(30))
display(frame(data.get("spending_categories")).head(30))